# ML Model Stress Test & Evaluation

This notebook evaluates the stability, performance, and resource usage (VRAM) of the VLM and LLM pipeline.

## 1. Environment & Connectivity Check
Verifies libraries, GPU, and vLLM server endpoints.

In [1]:
import os
import time
import json
import pandas as pd
import subprocess
import paddle
from paddleocr import PaddleOCRVL
from openai import OpenAI
from pathlib import Path

print("Checking Paddle version:", paddle.__version__)
if paddle.is_compiled_with_cuda():
    paddle.set_device('gpu')
    print("GPU detected and active.")
else:
    print("WARNING: GPU NOT DETECTED. Paddle using CPU.")

# Connectivity Check: vLLM VLM Server (port 8000)
try:
    vlm_pipeline = PaddleOCRVL(
        vl_rec_backend="vllm-server",
        vl_rec_server_url="http://localhost:8000/v1",
        vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL"
    )
    print("VLM Server (8000) connectivity: SUCCESS")
except Exception as e:
    print(f"VLM Server (8000) connectivity: FAILED - {e}")

# Connectivity Check: vLLM LLM Server (port 8001)
try:
    client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")
    client.models.list()
    print("LLM Server (8001) connectivity: SUCCESS")
except Exception as e:
    print(f"LLM Server (8001) connectivity: FAILED - {e}")

print("\nAll libraries and endpoints verified.")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-DocLayoutV3`.


Checking Paddle version: 3.3.0
GPU detected and active.


Creating model: ('PaddleOCR-VL-1.5-0.9B', None)


VLM Server (8000) connectivity: SUCCESS


[2026-04-14 06:57:54,805] [    INFO] _client.py:1025 - HTTP Request: GET http://localhost:8001/v1/models "HTTP/1.1 200 OK"


LLM Server (8001) connectivity: SUCCESS

All libraries and endpoints verified.


## 2. Logger & Monitor Setup
Defining the `StressTestManager` to track metrics, errors, and VRAM.

In [4]:
class StressTestManager:
    def __init__(self, test_name, log_csv="stress_test_log.csv"):
        self.test_name = test_name
        self.log_csv = log_csv
        self.results = []
        
    def get_vram_usage(self):
        """Extracts current VRAM usage using nvidia-smi."""
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,nounits,noheader"],
                encoding='utf-8'
            )
            return int(result.strip())
        except:
            return 0

    def log_run(self, file_name, model_name, start_time, end_time, status="success", error_msg="", output_file=""):
        duration = end_time - start_time
        vram = self.get_vram_usage()
        
        entry = {
            "test_scenario": self.test_name,
            "model_name": model_name,
            "file_name": file_name,
            "start_time": time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time)),
            "end_time": time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time)),
            "time_taken_sec": round(duration, 3),
            "vram_used_mb": vram,
            "status": status,
            "error": error_msg,
            "output_file": output_file
        }
        self.results.append(entry)
        
    def save_to_csv(self):
        df = pd.DataFrame(self.results)
        if os.path.exists(self.log_csv):
            df.to_csv(self.log_csv, mode='a', header=False, index=False)
        else:
            df.to_csv(self.log_csv, index=False)
        print(f"Results saved to {self.log_csv}")

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

DATASET_DIR = "../falcon_invoice_dataset"
invoice_files = sorted([os.path.join(DATASET_DIR, f) for f in os.listdir(DATASET_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))])
print(f"Dataset: {len(invoice_files)} images found.")

Dataset: 10 images found.


## 3. Scenario 1: VLM Alone - Stress Test (50 Runs)
Evaluates VLM stability and memory over a loop.

In [6]:
VLM_OUTPUT_DIR = "../stress_results_vlm"
ensure_dir(VLM_OUTPUT_DIR)

manager = StressTestManager("VLM_Inference_Loop")
ITERATIONS = 50

print(f"Starting VLM Stress Test ({ITERATIONS} iterations)...")

for i in range(ITERATIONS):
    # Cycle through available images
    img_path = invoice_files[i % len(invoice_files)]
    file_basename = os.path.basename(img_path)
    
    start_t = time.time()
    try:
        # Running prediction
        results = vlm_pipeline.predict(img_path)
        
        # Saving result
        output_filename = f"vlm_out_{i:03d}_{file_basename}.json"
        output_path = os.path.join(VLM_OUTPUT_DIR, output_filename)
        
        # Assuming results[0] has save_to_json method from previous vlm_llm_test.ipynb usage
        results[0].save_to_json(save_path=output_path)
        
        manager.log_run(file_basename, "PaddleOCR-VL", start_t, time.time(), "success", "", output_path)
    except Exception as e:
        manager.log_run(file_basename, "PaddleOCR-VL", start_t, time.time(), "failed", str(e))
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{ITERATIONS} iterations...")

manager.save_to_csv()
print("VLM Stress Test Complete.")

Starting VLM Stress Test (50 iterations)...


[2026-04-14 07:11:44,897] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,900] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,907] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,918] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,922] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,925] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,955] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:11:44,971] [    INFO] _client.py:1740 - HTTP Re

Processed 10/50 iterations...


[2026-04-14 07:12:04,983] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:04,989] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:04,997] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:04,998] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:05,003] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:05,008] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:05,044] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:05,060] [    INFO] _client.py:1740 - HTTP Re

Processed 20/50 iterations...


...

Processed 30/50 iterations...


[2026-04-14 07:12:44,997] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,005] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,010] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,021] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,023] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,027] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,056] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:12:45,074] [    INFO] _client.py:1740 - HTTP Re

Processed 40/50 iterations...


[2026-04-14 07:13:05,522] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,532] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,545] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,548] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,551] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,563] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,606] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:13:05,639] [    INFO] _client.py:1740 - HTTP Re

Processed 50/50 iterations...
Results saved to stress_test_log.csv
VLM Stress Test Complete.


## 4. Scenario 2: LLM Alone - Stress Test (50 Runs)
Passes VLM outputs to LLM for data extraction.

In [17]:
LLM_OUTPUT_DIR = "../stress_results_llm"
ensure_dir(LLM_OUTPUT_DIR)

manager = StressTestManager("LLM_Extraction_Loop")
vlm_jsons = sorted([os.path.join(VLM_OUTPUT_DIR, f) for f in os.listdir(VLM_OUTPUT_DIR) if f.endswith('.json')])

system_prompt = """You are a precise data extraction assistant. Extract information and return valid JSON per this schema:
{
  "Person_name": "string", "Company_name": "string", "address": "string", "contact": "string",
  "invoice_number": "string", "invoice_date": "YYYY-MM-DD", "due_date": "YYYY-MM-DD",
  "subtotal": "float", "tax": "float", "total": "float"
}"""

def get_raw_content(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    contents = []
    # Handle both list and dict results if structure varies
    block_list = data[0]['parsing_res_list'] if isinstance(data, list) else data['parsing_res_list']
    for block in block_list:
        if 'block_content' in block and block['block_content']:
            contents.append(block['block_content'])
    return "\n".join(contents)

print(f"Starting LLM Stress Test ({len(vlm_jsons)} VLM files found)...")

for i, json_path in enumerate(vlm_jsons[:50]): # Cap at 50 if more exist
    file_basename = os.path.basename(json_path)
    start_t = time.time()
    
    try:
        raw_text = get_raw_content(json_path)
        response = client.chat.completions.create(
            model="Qwen2.5-1.5B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text to process:\n{raw_text}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        llm_json = response.choices[0].message.content
        output_filename = file_basename.replace(".json", "_llm.txt")
        output_path = os.path.join(LLM_OUTPUT_DIR, output_filename)
        
        with open(output_path, 'w') as f:
            f.write(llm_json)
            
        manager.log_run(file_basename, "Qwen2.5-1.5B", start_t, time.time(), "success", "", output_path)
    except Exception as e:
        # print(e)
        manager.log_run(file_basename, "Qwen2.5-1.5B", start_t, time.time(), "failed", str(e))

    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{min(50, len(vlm_jsons))} iterations...")

manager.save_to_csv()
print("LLM Stress Test Complete.")

Starting LLM Stress Test (50 VLM files found)...


[2026-04-14 07:47:13,108] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:15,429] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:15,482] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-14 07:47:17,779] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:19,904] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:22,193] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:24,371] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:26,669] [    INFO] _client.py:1025 

Processed 10/50 iterations...


[2026-04-14 07:47:34,435] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:36,728] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:36,779] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-14 07:47:39,077] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:41,211] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:43,714] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:45,975] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:47:48,085] [    INFO] _client.py:1025 

Processed 20/50 iterations...


...

Processed 30/50 iterations...


[2026-04-14 07:48:16,971] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:19,282] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:19,321] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-14 07:48:21,613] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:23,769] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:26,061] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:28,336] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:30,646] [    INFO] _client.py:1025 

Processed 40/50 iterations...


[2026-04-14 07:48:38,013] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:40,470] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:40,523] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-14 07:48:42,820] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:44,958] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:47,487] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:49,748] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-14 07:48:52,059] [    INFO] _client.py:1025 

Processed 50/50 iterations...
Results saved to stress_test_log.csv
LLM Stress Test Complete.


In [17]:
LLM_OUTPUT_DIR = "../stress_results_llm"
# Assuming ensure_dir is defined elsewhere in your notebook
ensure_dir(LLM_OUTPUT_DIR)

# Assuming StressTestManager is defined elsewhere
manager = StressTestManager("LLM_Extraction_Loop")

# Assuming VLM_OUTPUT_DIR is defined elsewhere (e.g., VLM_OUTPUT_DIR = "stress_results_vlm")
vlm_jsons = sorted([os.path.join(VLM_OUTPUT_DIR, f) for f in os.listdir(VLM_OUTPUT_DIR) if f.endswith('.json')])

system_prompt = """You are a precise data extraction assistant. Extract information and return valid JSON per this schema:
{
  "Person_name": "string", "Company_name": "string", "address": "string", "contact": "string",
  "invoice_number": "string", "invoice_date": "YYYY-MM-DD", "due_date": "YYYY-MM-DD",
  "subtotal": "float", "tax": "float", "total": "float"
}"""


# No Crashes: The second version won't throw an error if the JSON file is empty or formatted slightly differently than expected.
# Cleaner Output: By using .strip(), the second version prevents your final string from having lines that contain only invisible spaces or tabs.
# Type Safety: It explicitly handles cases where block_content might not be a string (by casting with str()).


def get_raw_content(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    contents = []
    
    # Safely get the parsing_res_list from the top-level dictionary
    if isinstance(data, dict):
        block_list = data.get('parsing_res_list', [])
    elif isinstance(data, list) and len(data) > 0:
        block_list = data[0].get('parsing_res_list', [])
    else:
        block_list = []

    for block in block_list:
        if isinstance(block, dict):
            # Target the specific key found in your JSON: 'block_content'
            text_value = block.get('block_content')
            
            # Append if it exists and isn't just empty space
            if text_value and str(text_value).strip():
                contents.append(str(text_value).strip())
                
    return "\n".join(contents)

print(f"Starting LLM Stress Test ({len(vlm_jsons)} VLM files found)...")

for i, json_path in enumerate(vlm_jsons[:50]): # Cap at 50 if more exist
    file_basename = os.path.basename(json_path)
    start_t = time.time()
    
    try:
        raw_text = get_raw_content(json_path)
        
        # Make sure 'client' is properly initialized via the OpenAI SDK before this loop!
        response = client.chat.completions.create(
            model="Qwen2.5-1.5B",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text to process:\n{raw_text}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        llm_json = response.choices[0].message.content
        output_filename = file_basename.replace(".json", "_llm.txt")
        output_path = os.path.join(LLM_OUTPUT_DIR, output_filename)
        
        with open(output_path, 'w') as f:
            f.write(llm_json)
            
        manager.log_run(file_basename, "Qwen2.5-1.5B", start_t, time.time(), "success", "", output_path)
    except Exception as e:
        manager.log_run(file_basename, "Qwen2.5-1.5B", start_t, time.time(), "failed", str(e))

    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{min(50, len(vlm_jsons))} iterations...")

manager.save_to_csv()
print("LLM Stress Test Complete.")

Starting LLM Stress Test (50 VLM files found)...


[2026-04-10 09:25:53,540] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:25:55,834] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:25:55,909] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-10 09:25:58,197] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:00,321] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:02,614] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:04,809] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:06,924] [    INFO] _client.py:1025 

Processed 10/50 iterations...


[2026-04-10 09:26:14,594] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:17,030] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:17,068] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-10 09:26:19,347] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:21,270] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:23,364] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:25,605] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:27,903] [    INFO] _client.py:1025 

Processed 20/50 iterations...


...

Processed 30/50 iterations...


[2026-04-10 09:26:57,071] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:59,364] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:26:59,403] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-10 09:27:01,668] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:03,786] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:06,083] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:08,275] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:10,380] [    INFO] _client.py:1025 

Processed 40/50 iterations...


[2026-04-10 09:27:18,074] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:20,520] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:20,572] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 400 Bad Request"
[2026-04-10 09:27:22,870] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:25,006] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:27,523] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:29,709] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-10 09:27:32,000] [    INFO] _client.py:1025 

Processed 50/50 iterations...
Results saved to stress_test_log.csv
LLM Stress Test Complete.


## 5. Scenario 3: Pipeline Lifecycle + VLM Stress Test (50 Runs)
Initializes pipeline AND runs inference in each loop to check for memory leaks during creation.

In [ ]:
LIFECYCLE_OUTPUT_DIR = "../stress_results_lifecycle"
ensure_dir(LIFECYCLE_OUTPUT_DIR)

manager = StressTestManager("Pipeline_Lifecycle_Loop", log_csv="pipeline_stress_log.csv")
ITERATIONS = 50

print(f"Starting Lifecycle Stress Test ({ITERATIONS} iterations)...")

for i in range(ITERATIONS):
    img_path = invoice_files[i % len(invoice_files)]
    file_basename = os.path.basename(img_path)
    start_t = time.time()
    
    try:
        # 1. Initialize Pipeline
        temp_pipeline = PaddleOCRVL(
            vl_rec_backend="vllm-server",
            vl_rec_server_url="http://localhost:8000/v1",
            vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL"
        )
        
        # 2. Predict
        results = temp_pipeline.predict(img_path)
        
        # 3. Save
        output_path = os.path.join(LIFECYCLE_OUTPUT_DIR, f"lifecycle_out_{i:03d}.json")
        results[0].save_to_json(save_path=output_path)
        
        # 4. Explicitly delete (help GC if possible, though Paddle objects may persist in C++ memory)
        del temp_pipeline
        
        manager.log_run(file_basename, "PaddleOCR-VL_FullLifecycle", start_t, time.time(), "success", "", output_path)
    except Exception as e:
        manager.log_run(file_basename, "PaddleOCR-VL_FullLifecycle", start_t, time.time(), "failed", str(e))
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{ITERATIONS} iterations...")

manager.save_to_csv()
print("Lifecycle Stress Test Complete.")

## 6. Summary Statistics & Error Rates

In [19]:
if os.path.exists("stress_test_log.csv"):
    df = pd.read_csv("stress_test_log.csv")
    
    summary = df.groupby(['test_scenario', 'status']).agg({
        'time_taken_sec': ['mean', 'max', 'min', 'count'],
        'vram_used_mb': ['mean', 'max']
    })
    
    print("--- STRESS TEST SUMMARY ---")
    display(summary)
    
    error_counts = df[df['status'] == 'failed'].groupby('test_scenario').size()
    total_counts = df.groupby('test_scenario').size()
    error_rate = (error_counts / total_counts * 100).fillna(0)
    
    print("\n--- ERROR RATES (%) ---")
    print(error_rate)
else:
    print("No log file found. Ensure scenarios have run.")

--- STRESS TEST SUMMARY ---


time_taken_sec                     vram_used_mb  \
                                      mean    max    min count         mean   
test_scenario       status                                                    
LLM_Extraction_Loop failed        0.019600  0.042  0.012    10     12941.00   
                    success       2.317489  2.844  1.917    90     12941.00   
VLM_Inference_Loop  success       2.012320  3.952  1.534   100     12940.52   

                                    
                               max  
test_scenario       status          
LLM_Extraction_Loop failed   12941  
                    success  12941  
VLM_Inference_Loop  success  12941


--- ERROR RATES (%) ---
test_scenario
LLM_Extraction_Loop    10.0
VLM_Inference_Loop      0.0
dtype: float64
